# ResQRoute - crops + training on Kaggle

Inputs (no internet needed):
- `tunguz/xview2-challenge-dataset-train-and-test` - xBD images
- `resqroute-code` - the repo's `scripts/*.py` + `outputs/` manifests, uploaded flat
- `resqroute-imagenet-weights` - torchvision ImageNet checkpoints (resnet18, mobilenet_v3_small, efficientnet_b0)

Uses a GPU if one is attached, otherwise trains on CPU (slower, same results).
Outputs: `models_outputs.zip` (`models/` + `outputs/`), and `crops.zip` (`data/crops/`) when `ZIP_CROPS=True`.

In [ ]:
import os, shutil, subprocess, sys, pathlib
inp = pathlib.Path('/kaggle/input')
code_dir = next(inp.rglob('common.py')).parent
work = pathlib.Path('/kaggle/working/resqroute')
(work / 'scripts').mkdir(parents=True, exist_ok=True); (work / 'outputs').mkdir(exist_ok=True)
for f in code_dir.iterdir():
    if f.suffix == '.py': shutil.copy(f, work / 'scripts')
    elif f.suffix in ('.csv', '.json') and f.name != 'dataset-metadata.json': shutil.copy(f, work / 'outputs')

# torchvision looks for pretrained weights in $TORCH_HOME/hub/checkpoints - serve them from the uploaded dataset
ckpt = pathlib.Path('/kaggle/working/torch/hub/checkpoints'); ckpt.mkdir(parents=True, exist_ok=True)
for f in inp.rglob('*.pth'):
    shutil.copy(f, ckpt)
os.environ['TORCH_HOME'] = '/kaggle/working/torch'
os.environ['RESQROUTE_ROOT'] = str(work)
os.chdir(work)

def run(*args):
    # stream output and stop the notebook on failure (a plain `!cmd` would not)
    print('$', ' '.join(args), flush=True)
    subprocess.run([sys.executable, *args], check=True)

def zip_dirs(zip_path, dirs):
    import zipfile
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for d in dirs:
            for f in sorted((work / d).rglob('*')):
                if f.is_file(): z.write(f, f.relative_to(work))

import torch
print('code from', code_dir, '->', sorted(os.listdir(work / 'outputs')))
print('imagenet weights:', sorted(os.listdir(ckpt)))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (training on CPU)', '| CPUs:', os.cpu_count())

In [ ]:
# Step 3: cut 224x224 pre/post crops (deterministic; set ZIP_CROPS=True to also save them as crops.zip)
ZIP_CROPS = False
run('scripts/03_extract_crops.py', '--workers', '4')
if ZIP_CROPS:
    zip_dirs('/kaggle/working/crops.zip', ['data'])

In [ ]:
# Step 4: fine-tune the three classifiers (3-class); early stopping on val loss
workers = '4' if torch.cuda.is_available() else '2'
for arch in ['mobilenet_v3_small', 'resnet18', 'efficientnet_b0']:
    run('scripts/04_train_model.py', '--arch', arch, '--workers', workers)
    zip_dirs('/kaggle/working/models_outputs.zip', ['models', 'outputs'])  # refreshed after each model

In [ ]:
# keep the output small: the zips hold everything
zip_dirs('/kaggle/working/models_outputs.zip', ['models', 'outputs'])
shutil.rmtree(work); shutil.rmtree('/kaggle/working/torch')
print(sorted((f.name, round(f.stat().st_size / 1e6, 1)) for f in pathlib.Path('/kaggle/working').iterdir()))